 ##### Model 1.2


Did this hitter strike out against this pitcher in this game?
    0 = no strikeout
    1 = at least one strikeout

1. Imports
2. Connect to SQL Server
3. Load model table
4. Inspect data
5. Build target
6. Drop leakage columns
7. Prepare X and y
8. Train/test split
9. Train XGBoost classifier
10. Evaluate model
11. Predict probabilities
12. Aggregate to pitcher expected Ks

In [58]:
import pandas as pd
from sqlalchemy import create_engine

SERVER = "localhost"
DATABASE = "mlb"
DRIVER = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

query = """
SELECT *
FROM mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nFirst 20 columns:")
print(df.columns[:20].tolist())

Shape: (62220, 543)

First 5 rows:


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,lineup_weighted_whiff_rate_vs_rhp_last_3,lineup_weighted_whiff_rate_vs_rhp_last_5,lineup_weighted_whiff_rate_vs_rhp_last_10,lineup_weighted_whiff_rate_vs_lhp_last_3,lineup_weighted_whiff_rate_vs_lhp_last_5,lineup_weighted_whiff_rate_vs_lhp_last_10,lineup_num_high_k_hitters,lineup_num_power_hitters,lineup_num_high_whiff_hitters,lineup_num_strong_bats
0,776680,2025-08-19,2025,647304,Josh Naylor,1B,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
1,776680,2025-08-19,2025,668227,Randy Arozarena,LF,136,Seattle Mariners,661395,Jhoan Duran,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
2,776680,2025-08-19,2025,668227,Randy Arozarena,LF,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
3,776680,2025-08-19,2025,670042,Luke Raley,RF,136,Seattle Mariners,661395,Jhoan Duran,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
4,776680,2025-08-19,2025,553993,Eugenio Suárez,3B,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3



First 20 columns:
['gamePk', 'game_date', 'season', 'hitter_id', 'hitter_name', 'hitter_position', 'hitter_team_id', 'hitter_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_throws', 'hitter_stand', 'hitter_strikeOuts', 'pitches_seen_vs_pitcher', 'swings_vs_pitcher', 'whiffs_vs_pitcher', 'called_strikes_vs_pitcher', 'matchup_whiff_rate']


In [59]:
# Convert hitter_strikeOuts to binary target
df["target_hitter_k"] = (df["hitter_strikeOuts"] > 0).astype(int)

print(df["target_hitter_k"].value_counts())
print(df["target_hitter_k"].value_counts(normalize=True))

target_hitter_k
0    42120
1    20100
Name: count, dtype: int64
target_hitter_k
0    0.676953
1    0.323047
Name: proportion, dtype: float64


In [60]:
# Define leakage columns - Ensure we clean it by ensuring that the model does not see actual results which is cheating.
leakage_cols_direct = [
    "hitter_strikeOuts",          # source of target
    "pitcher_strikeOuts",         # future info
    "pitches_seen_vs_pitcher",
    "swings_vs_pitcher",
    "whiffs_vs_pitcher",
    "called_strikes_vs_pitcher",
    "matchup_whiff_rate",
    "matchup_called_strike_rate",
    "matchup_csw_rate"
]

In [61]:
# Dropping additional leakage columns that already has the game results
leakage_cols_pattern = [
    col for col in df.columns
    if (
        "hitter_pitcher_" in col
        or "hitter_game_" in col
    )
]

In [62]:
df_ids = df[[
    "gamePk",
    "game_date",
    "pitcher_id",
    "pitcher_name",
    "hitter_id",
    "pitcher_team_name"
]].copy()

In [63]:
drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_team_name",
    "pitcher_team_name"
]

In [64]:
cols_to_drop = list(set(leakage_cols_direct + leakage_cols_pattern + drop_cols))

df_model = df.drop(columns=cols_to_drop, errors="ignore")

print("Original shape:", df.shape)
print("Model shape:", df_model.shape)

Original shape: (62220, 544)
Model shape: (62220, 502)


In [65]:
# Check that obvious leakage is gone
still_bad = [
    col for col in df_model.columns
    if (
        col in leakage_cols_direct
        or "hitter_pitcher_" in col
        or "hitter_game_" in col
    )
]

print("Remaining suspicious columns:", still_bad[:50])
print("Count remaining suspicious columns:", len(still_bad))

Remaining suspicious columns: []
Count remaining suspicious columns: 0


In [66]:
# Split by season, not randomly
train_df = df_model[df_model["season"] == 2025].copy()
test_df  = df_model[df_model["season"] == 2026].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain target distribution:")
print(train_df["target_hitter_k"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["target_hitter_k"].value_counts(normalize=True))

Train shape: (47346, 502)
Test shape: (14874, 502)

Train target distribution:
target_hitter_k
0    0.673953
1    0.326047
Name: proportion, dtype: float64

Test target distribution:
target_hitter_k
0    0.6865
1    0.3135
Name: proportion, dtype: float64


In [67]:
# Build X and y
X_train = train_df.drop(columns=["target_hitter_k"])
y_train = train_df["target_hitter_k"]

X_test = test_df.drop(columns=["target_hitter_k"])
y_test = test_df["target_hitter_k"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (47346, 501)
X_test: (14874, 501)
y_train: (47346,)
y_test: (14874,)


In [68]:
# Check data types
print(X_train.dtypes.value_counts())

object_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
print("Object columns:", object_cols)

float64    484
int64       12
str          5
Name: count, dtype: int64
Object columns: ['hitter_position', 'pitcher_throws', 'hitter_stand', 'hitter_lineup_position', 'hitter_lineup_position_name']


In [69]:
# Encode categorical columns after the split
from sklearn.preprocessing import LabelEncoder

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

label_encoders = {}

for col in object_cols:
    le = LabelEncoder()

    X_train_encoded[col] = le.fit_transform(X_train_encoded[col].astype(str))

    X_test_encoded[col] = X_test_encoded[col].map(
        lambda s: le.transform([str(s)])[0] if str(s) in le.classes_ else -1
    )

    label_encoders[col] = le

print("Encoding complete ✅")
print(X_train_encoded.dtypes.value_counts())
print(X_test_encoded.dtypes.value_counts())

Encoding complete ✅
float64    484
int64       17
Name: count, dtype: int64
float64    484
int64       17
Name: count, dtype: int64


In [70]:
# Train the classifier
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_encoded, y_train)

print("Model trained ✅")

Model trained ✅


In [71]:
# Predict on 2026
y_pred = model.predict(X_test_encoded)
y_prob = model.predict_proba(X_test_encoded)[:, 1]

In [72]:
# Evaluate the model
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("Log Loss:", log_loss(y_test, y_prob))

Accuracy: 0.6873739411052844
ROC AUC: 0.6332602440216346
Log Loss: 0.5999167682083649


In [73]:
# Check feature importance
feature_importance = pd.DataFrame({
    "feature": X_train_encoded.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

display(feature_importance.head(20))

,feature,importance
280,pitcher_gamesStarted,0.072326
300,pitcher_avg_games_started_last_5,0.060622
284,pitcher_avg_bf_last_3,0.015924
89,hitter_avg_k_last_10,0.009062
372,pitcher_prev_pitches,0.006870
118,hitter_weighted_k_rate_last_10,0.005482
250,hitter_weighted_whiff_rate_last_10,0.005281
323,pitcher_avg_k_last_10,0.004855
236,hitter_avg_whiff_rate_last_10,0.004610
237,hitter_avg_contact_rate_last_10,0.004281


In [74]:
# Build prediction table  After predictions, merge IDs back
pred_df = test_df.copy()
pred_df["pred_hitter_k_prob"] = y_prob

# bring IDs back
pred_df = pred_df.merge(
    df_ids,
    on=["pitcher_id", "hitter_id"],
    how="left"
)

In [76]:
# Aggregate to pitcher level
pitcher_expected_k = (
    pred_df.groupby(
        ["gamePk", "game_date", "pitcher_id", "pitcher_name", "pitcher_team_name"],
        as_index=False
    )
    .agg(
        expected_strikeouts=("pred_hitter_k_prob", "sum")
    )
)

display(pitcher_expected_k.head())

,gamePk,game_date,pitcher_id,pitcher_name,pitcher_team_name,expected_strikeouts
0,776135,2025-09-28,666171,Ryan Zeferjahn,Los Angeles Angels,0.296652
1,776135,2025-09-28,696147,Sam Bachman,Los Angeles Angels,0.513901
2,776136,2025-09-28,688158,David Morgan,San Diego Padres,0.202991
3,776138,2025-09-28,547184,Michael Kelly,Athletics,0.214552
4,776144,2025-09-28,605130,Scott Barlow,Athletics,0.227416


In [78]:
actual_k = df[[
    "gamePk",
    "game_date",
    "pitcher_id",
    "pitcher_name",
    "pitcher_team_name",
    "pitcher_strikeOuts"
]].drop_duplicates()

print(actual_k.shape)
display(actual_k.head())

(13328, 6)


,gamePk,game_date,pitcher_id,pitcher_name,pitcher_team_name,pitcher_strikeOuts
0,776680,2025-08-19,650911,Cristopher Sánchez,Philadelphia Phillies,12
1,776680,2025-08-19,661395,Jhoan Duran,Philadelphia Phillies,1
6,776683,2025-08-18,695243,Mason Miller,San Diego Padres,1
7,776683,2025-08-18,593974,Wandy Peralta,San Diego Padres,1
8,776755,2025-08-13,683232,Nick Mears,Kansas City Royals,1


In [79]:
pitcher_expected_k = pitcher_expected_k.merge(
    actual_k,
    on=["gamePk", "game_date", "pitcher_id", "pitcher_name", "pitcher_team_name"],
    how="left"
)

print(pitcher_expected_k.shape)
display(pitcher_expected_k.head())

(5058, 7)


,gamePk,game_date,pitcher_id,pitcher_name,pitcher_team_name,expected_strikeouts,pitcher_strikeOuts
0,776135,2025-09-28,666171,Ryan Zeferjahn,Los Angeles Angels,0.296652,1
1,776135,2025-09-28,696147,Sam Bachman,Los Angeles Angels,0.513901,0
2,776136,2025-09-28,688158,David Morgan,San Diego Padres,0.202991,2
3,776138,2025-09-28,547184,Michael Kelly,Athletics,0.214552,1
4,776144,2025-09-28,605130,Scott Barlow,Athletics,0.227416,1


In [80]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(
    pitcher_expected_k["pitcher_strikeOuts"],
    pitcher_expected_k["expected_strikeouts"]
)

rmse = np.sqrt(mean_squared_error(
    pitcher_expected_k["pitcher_strikeOuts"],
    pitcher_expected_k["expected_strikeouts"]
))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.461612582206726
RMSE: 2.1660693396743658


#### Convert to probaility - 

In [81]:
# posisson distribution

from scipy.stats import poisson

pitcher_expected_k["line"] = 5.5  # example line

pitcher_expected_k["prob_over"] = 1 - poisson.cdf(
    pitcher_expected_k["line"],
    pitcher_expected_k["expected_strikeouts"]
)

In [82]:
# add the odds
pitcher_expected_k["odds_over"] = 1.90

In [ ]:
# implied probability
pitcher_expected_k["implied_prob"] = 1 / pitcher_expected_k["odds_over"]

In [ ]:
# Expected Value (EV)
pitcher_expected_k["ev"] = (
    pitcher_expected_k["prob_over"] * pitcher_expected_k["odds_over"]
) - 1

In [85]:
# Betting signal
pitcher_expected_k["bet_over"] = pitcher_expected_k["ev"] > 0